- Bước 1: Load video lên  
- Bước 2: Tách theo từng frame và duyệt theo từng frame  
    - Bước 2.1: Chạy detect trên frame i  
    - Bước 2.2: Cắt từng box output của frame i  
        - Bước 2.2.1: Chạy extract feature trên từng box j của frame i  
        - Bước 2.2.2: Lấy feature vector của các box j của frame i  
    - Bước 2.3: Chạy tracker kết nối frame i với các frame i-1, i-2 bằng kết quả detect/appearance feature tương ứng với mỗi loại tracker.  
- Bước 3: Offline refinement với GTALink, trả về danh sách tracklet id, frame và vị trí tại mỗi frame.
- Bước 4: Visualize.

In [15]:
import cv2
import os
import numpy as np
from typing import List, Dict, Optional
from src.feature_extractor import FeatureExtractor
from src.detector import Detector
from src.tracker import Tracker
from src.gtalink import GTALink
from src.compute_metrics import compute_metrics

# Path

In [16]:
YOLO_PATH    = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\PipelineMOT\models\best_yolo26.pt"
RF_PATH      = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\PipelineMOT\models\best_rf_detr.pth"
SOLIDER_PATH = r"D:\UITs subject\Năm 3\Nhận dạng\SOLIDER\checkpoints\swin_small_converted.pth"
VIDEO_PATH   = r"D:\UITs subject\Năm 3\Nhận dạng\project\Data\wide_view\videos\F_20220220_1_1800_1830.mp4"
GT_CSV_PATH  = r"D:\UITs subject\Năm 3\Nhận dạng\project\Data\wide_view\annotations\F_20220220_1_1800_1830.csv"
OUTPUT_DIR   = r"D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ALL_DETECTOR  = ["yolo26", "rf_detr"]
ALL_EXTRACTOR = ["osnet", "solider", "color_histogram"]
ALL_TRACKER   = ["bytetrack", "ocsort", "strongsort", "deepeiou"]

# PIPELINE FUNCTIONS

In [17]:
_PALETTE = [
    (255, 56, 56),  (255, 157, 56), (255, 225, 56), (138, 232, 58),
    (56, 232, 124), (56, 232, 225), (56, 128, 255), (174, 56, 255),
    (255, 56, 159), (212, 212, 212),(255, 130, 77), (77, 255, 130),
    (77, 130, 255), (255, 77, 130), (130, 255, 77), (130, 77, 255),
]

def _color(track_id: int):
    return _PALETTE[track_id % len(_PALETTE)]

In [18]:
def run_mot(video_path, detector, tracker, extractor, refiner = None, extractor_refine = None):
    """
    Chạy full MOT pipeline cho một video.
 
    Parameters
    ----------
    video_path : str
    detector   : object với method .detect(frame) → [[x1,y1,x2,y2,conf], ...]
    tracker    : Tracker instance
    extractor  : object với method .extract(crop) → np.ndarray (L2-normalised), dùng cho online tracking
    refiner    : (optional) object với .refine(all_tracks) → refined_tracks
                 Nếu None, trả về all_tracks trực tiếp.
    extractor_refine: (optional) object với method .extract(crop) → np.ndarray
                       Extractor riêng để tạo embedding cho refinement.
                       Nếu None nhưng refiner được truyền vào → dùng chung với online tracking
 
    Returns
    -------
    all_tracks : dict
        {
            track_id (int): {
                'boxes' : list — box [x1,y1,x2,y2] hoặc None theo từng frame,
                'frames': list — frame_idx tương ứng (chỉ các frame có box)
            }
        }
    """
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")
    
    # all_tracks[track_id]['boxes'][i] = box tại frame i, hoặc None nếu không có
    all_tracks: Dict[int, Dict] = {} 

    frame_idx = 0   # 0-based index để index vào list
    frame_id  = 1   # 1-based id truyền vào tracker (convention của STrack)
    
    # extractor của refiner
    extractor_refine = extractor_refine if (refiner is not None and extractor_refine is not None) else extractor
    
    # Duyệt qua từng frame 
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        print(f"Processing frame {frame_idx}")
        
        # Bước 2.1: Detect
        detections = detector.detect(frame) # Trả về list các box và conf [[x1, y1, x2, y2, conf]]
        
        # Bước 2.2: Crop + Feature
        enriched_detections = [] # Tổng hợp box, conf, feature tại frame đang xét của các object
        for det in detections:
            enriched = {}
            x1, y1, x2, y2 = map(int, det[:-1])
            score = float(det[4])
            
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            # 2.2.1 Extract feature
            feature = extractor.extract(crop)

            # 2.2.2 attach feature
            enriched_detections.append({
                'tlbr' : [float(x1), float(y1), float(x2), float(y2)],
                'score': score,
                'feat' : feature,
            })

        # Bước 2.3: Tracking
        active_tracks  = tracker.update(enriched_detections, frame_id) 
         
        # Lưu all_tracks
        # Với các track_id mới: chèn None cho tất cả frame trước đó
        active_ids = set()
        for track in active_tracks:
            tid = track.track_id
            active_ids.add(tid)
 
            if tid not in all_tracks:
                # Track mới → fill None cho các frame trước
                entry = {
                    'boxes' : [None] * frame_idx,
                    'frames': [],
                }
                if refiner is not None:
                    entry['feats'] = []         # chỉ khởi tạo khi cần refinement
                all_tracks[tid] = entry
 
            box = track.tlbr.tolist()   # [x1,y1,x2,y2] 
            all_tracks[tid]['boxes'].append(box)
            all_tracks[tid]['frames'].append(frame_idx)
 
            # Nếu cần refine:
            if refiner is not None:
                    x1, y1, x2, y2 = map(int, track.tlbr)
                    # Clamp để tránh ra ngoài biên frame
                    crop = frame[y1:y2, x1:x2]
                    feat = extractor_refine.extract(crop) if crop.size > 0 \
                        else extractor_refine.extract(frame[0:1, 0:1])  # fallback crop rỗng
                    all_tracks[tid]['feats'].append(feat)
                    
        # Track đã có trong dict nhưng không active frame này → None
        for tid, data in all_tracks.items():
            if tid not in active_ids:
                # Chỉ thêm None nếu track chưa có entry cho frame này
                if len(data['boxes']) == frame_idx:
                    data['boxes'].append(None)
 
        frame_idx += 1
        frame_id  += 1
        
    cap.release()
    print(f"\n[MOT] Done — {frame_idx} frames, {len(all_tracks)} tracks")

    # Bước 3: Offline refinement
    if refiner is not None:
        return refiner.refine(all_tracks)   #all_tracks có feat nhưng bỏ feat sau output refiner
    return all_tracks

In [19]:
def visualize_tracks(video_path: str,
                     all_tracks: Dict,
                     output_path: str = "output_tracked.avi",  # ← Dùng .avi
                     show_id: bool = True,
                     thickness: int = 2,
                     font_scale: float = 0.7):
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Không mở được video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Buộc dùng AVI + MJPG (tương thích cao nhất với Windows)
    if not output_path.lower().endswith('.avi'):
        output_path = output_path.rsplit('.', 1)[0] + '.avi'

    fourcc = cv2.VideoWriter_fourcc(*'MJPG')      # Codec dễ mở nhất
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not writer.isOpened():
        print("⚠️ MJPG không hoạt động, thử XVID...")
        fourcc = cv2.VideoWriter_fourcc(*'XVID')
        writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Xây frame map
    frame_map: Dict[int, Dict[int, List]] = {}
    for tid, data in all_tracks.items():
        for fi, box in enumerate(data.get('boxes', [])):
            if box is not None:
                frame_map.setdefault(fi, {})[tid] = box

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        for tid, box in frame_map.get(frame_idx, {}).items():
            x1, y1, x2, y2 = map(int, box)
            color = _color(tid)

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

            if show_id:
                label = f"ID {tid}"
                (tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
                label_y = max(y1 - 5, th + 5)

                cv2.rectangle(frame, 
                              (x1, label_y - th - baseline - 4),
                              (x1 + tw + 6, label_y + 2), 
                              color, cv2.FILLED)
                
                cv2.putText(frame, label, (x1 + 3, label_y - baseline),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (255,255,255), thickness, cv2.LINE_AA)

        cv2.putText(frame, f"Frame {frame_idx+1}/{total}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (220,220,220), 2, cv2.LINE_AA)

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"✅ Đã lưu xong: {output_path}")
    print("   → Hãy thử mở file này bằng Windows Media Player")

In [20]:
def run_and_visualize(video_path: str,
                      detector,
                      extractor,
                      output_path: str = 'output_tracked.avi',
                      algorithm: str = 'deepeiou',
                      refiner=None,
                      refiner_extractor = None,
                      tracker_kwargs: Optional[Dict] = None,
                      **viz_kwargs) -> Dict:
    
    tracker_kwargs = tracker_kwargs or {}
    tracker = Tracker(algorithm=algorithm, **tracker_kwargs)
 
    all_tracks = run_mot(
        video_path=video_path,
        detector=detector,
        tracker=tracker,
        extractor=extractor,
        refiner=refiner,
        extractor_refine=refiner_extractor
    )
 
    visualize_tracks(
        video_path=video_path,
        all_tracks=all_tracks,
        output_path=output_path,
        **viz_kwargs,
    )
 
    return all_tracks   # lấy về để tính theo metrics so sánh

In [21]:
def run_pipeline(
    video_path:        str,
    gt_csv_path:       str,
    detector,
    tracker_name:      str,
    output_path:       str,
    extractor=None,                # online extractor (DeepEIoU dùng, ByteTrack bỏ qua)
    refiner=None,
    refiner_extractor=None,        # extractor cho GTALink
    iou_thresh:        float = 0.5,
    tracker_kwargs:    Optional[Dict] = None,
    **viz_kwargs
) -> Dict:
    """
    Chạy full pipeline MOT và tính metrics.

    Parameters
    ----------
    video_path        : đường dẫn video
    gt_csv_path       : đường dẫn file annotation CSV (để tính metrics)
    detector          : Detector instance
    tracker_name      : tên tracker ("deepeiou" | "bytetrack" | "ocsort" | "strongsort")
    output_path       : đường dẫn lưu video output
    extractor         : extractor cho online tracking (None nếu tracker không dùng)
    refiner           : GTALink instance (None nếu không dùng refinement)
    refiner_extractor : extractor cho GTALink (None nếu không dùng refinement)
    iou_thresh        : IoU threshold để tính metrics
    tracker_kwargs    : kwargs truyền vào tracker constructor

    Returns
    -------
    dict với all_tracks và metrics
    """
    tracker_kwargs = tracker_kwargs or {}
    tracker        = Tracker(algorithm=tracker_name, **tracker_kwargs)

    print(f"\n{'='*60}")
    print(f"PIPELINE: {detector.backend.upper()} → "
          f"{(extractor.backend if extractor else 'None').upper()} → "
          f"{tracker_name.upper()} → "
          f"GTALink({refiner_extractor.backend if refiner_extractor else 'None'})")
    print(f"{'='*60}")

    # Bước 1-3: MOT
    all_tracks = run_mot(
        video_path      = video_path,
        detector        = detector,
        tracker         = tracker,
        extractor       = extractor,
        refiner         = refiner,
        extractor_refine= refiner_extractor,
    )

    # Bước 4: Visualize
    visualize_tracks(
        video_path  = video_path,
        all_tracks  = all_tracks,
        output_path = output_path,
        **viz_kwargs,
    )

    # Tính metrics
    metrics = compute_metrics(
        all_tracks = all_tracks,
        csv_path   = gt_csv_path,
        iou_thresh = iou_thresh,
    )

    return {'all_tracks': all_tracks, 'metrics': metrics}

# KHỞI TẠO COMPONENTS

In [23]:
detector          = Detector(backend="yolo26", model_path=YOLO_PATH)
extractor_color   = FeatureExtractor(backend="color_histogram")
extractor_solider = FeatureExtractor(
    backend="solider",
    solider_model_path=SOLIDER_PATH,
    solider_arch="swin_small",
    solider_semantic_weight=0.2,
)

[Detector] YOLO loaded on cuda
Missing: 0 | Unexpected: 8
[FeatureExtractor] SOLIDER (swin_small) loaded on cuda
[FeatureExtractor] Feature dim: 768


### Pipeline 1: YOLO26 + Color + DeepEIoU + GTALink(Color)

In [24]:
result_1 = run_pipeline(
    video_path        = VIDEO_PATH,
    gt_csv_path       = GT_CSV_PATH,
    detector          = detector,
    tracker_name      = "deepeiou",
    output_path       = os.path.join(OUTPUT_DIR, "yolo26_color_deepeiou_gtalink.avi"),
    extractor         = extractor_color,
    refiner           = GTALink(),
    refiner_extractor = extractor_color,
)



PIPELINE: YOLO26 → COLOR_HISTOGRAM → DEEPEIOU → GTALink(color_histogram)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame

Splitting tracklets: 100%|██████████| 48/48 [00:00<00:00, 140.09it/s]


[GTALink] Sau  split: 63 tracklets
[GTALink] Trước merge: 63 tracklets
[GTALink] Sau  merge: 42 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\yolo26_color_deepeiou_gtalink.avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 15778 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    66.34 %
  MOTP  :    65.41 %
  IDF1  :    74.42 %
  HOTA  :    26.42 %
  IDs   :      310
  FP    :     2261
  FN    :     2983
  GT    :    16500


### Pipeline 2: YOLO26 + ByteTrack(IoU only) + GTALink(Solider)

In [26]:
result_2 = run_pipeline(
    video_path        = VIDEO_PATH,
    gt_csv_path       = GT_CSV_PATH,
    detector          = detector,
    tracker_name      = "bytetrack",
    output_path       = os.path.join(OUTPUT_DIR, "yolo26_bytetrack_gtalink_solider.avi"),
    extractor         = extractor_color,             # ByteTrack không dùng appearance
    refiner           = GTALink(),
    refiner_extractor = extractor_solider,
)


PIPELINE: YOLO26 → COLOR_HISTOGRAM → BYTETRACK → GTALink(solider)
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
Processing frame 30
Processing frame 31
Processing frame 32
Processing frame 33
Processing frame 34
Processing frame 35
Processing frame 36
Processing frame 37
Processing frame 38
Processing frame 39
Processing frame 40
Processing frame 41
Processing frame 42
Processing frame 43
Processing frame 44
Processing frame 45
Processing frame 46
Pro

Splitting tracklets: 100%|██████████| 59/59 [00:00<00:00, 97.87it/s]


[GTALink] Sau  split: 64 tracklets
[GTALink] Trước merge: 64 tracklets
[GTALink] Sau  merge: 25 tracklets
✅ Đã lưu xong: D:\UITs subject\Năm 3\Nhận dạng\CS318_RECOGNITION_SOCCERTRACK_MOT\output\yolo26_bytetrack_gtalink_solider.avi
   → Hãy thử mở file này bằng Windows Media Player
Đang đọc Ground Truth...
GT : 750 frames, 16500 objects
Pred: 750 frames, 14852 objects
Tính MOTA / MOTP / IDs...
Tính IDF1...
Tính HOTA...

       KẾT QUẢ ĐÁNH GIÁ MOT PIPELINE
  MOTA  :    71.56 %
  MOTP  :    65.32 %
  IDF1  :    71.77 %
  HOTA  :    34.17 %
  IDs   :        4
  FP    :     1520
  FN    :     3168
  GT    :    16500


# SO SÁNH KẾT QUẢ

In [27]:
print("\n" + "="*65)
print(f"{'Pipeline':<40} {'MOTA':>6} {'MOTP':>6} {'IDF1':>6} {'HOTA':>6} {'IDs':>5}")
print("="*65)
pipelines = {
    "YOLO+Color+DeepEIoU+GTALink(Color)"    : result_1['metrics'],
    "YOLO+ByteTrack+GTALink(Solider)"        : result_2['metrics'],
}
for name, m in pipelines.items():
    print(f"{name:<40} {m['MOTA']:>6.1f} {m['MOTP']:>6.1f} "
          f"{m['IDF1']:>6.1f} {m['HOTA']:>6.1f} {m['IDs']:>5d}")
print("="*65)


Pipeline                                   MOTA   MOTP   IDF1   HOTA   IDs
YOLO+Color+DeepEIoU+GTALink(Color)         66.3   65.4   74.4   26.4   310
YOLO+ByteTrack+GTALink(Solider)            71.6   65.3   71.8   34.2     4
